In [2]:
import robotic as ry
import numpy as np
import time

In [ ]:
"""
Proof-of-concept: spawn N blocks at Gaussian-sampled heights and let them fall


This script creates a table, spawns blocks with Z sampled from a normal
distribution, and runs the physx simulation.
"""


def spawn_blocks(C, n_blocks=10, mean_z=1.0, std_z=0.1,
                 x_range=(-0.4, 0.4), y_range=(-0.4, 0.4),
                 block_size=(0.06, 0.06, 0.06, 0.01), seed=None):
    """Add `n_blocks` to Config C. Z heights are sampled from N(mean_z, std_z).

    Arguments:
        C: ry.Config()
        n_blocks: how many blocks to spawn
        mean_z, std_z: gaussian parameters (meters)
        x_range, y_range: uniform sampling range for X/Y
        block_size: shape size for ry.ST.ssBox (lengths + roundness)
        seed: RNG seed for reproducibility
    Returns:
        list of block frame names
    """
    if seed is not None:
        np.random.seed(seed)

    # table height
    table = C.getFrame('table')
    table_z = table.getPosition()[2]
    half_height = block_size[2] / 2.0
    min_z = table_z + half_height + 0.01

    names = []
    zs = np.random.normal(loc=mean_z, scale=std_z, size=n_blocks)
    for i in range(n_blocks):
        name = f'block_{i}'
        x = float(np.random.uniform(*x_range))
        y = float(np.random.uniform(*y_range))
        z = float(max(zs[i], min_z))  # avoid spawning inside the table

        f = C.addFrame(name)
        f.setShape(ry.ST.ssBox, size=list(block_size))
        f.setPosition([x, y, z])
        f.setColor([0.7, 0.3 + 0.02*i, 0.2 + 0.05*i]) # SIMPLE RANDOM SAMPLING FOR COLORS CAN BE IMPROVED LATER
        f.setMass(0.2)
        f.setContact(True)
        # MAYBE SET FRICTION AND OTHER PARAMS LATTER
        names.append(name)

    return names

In [ ]:
def main():
    # parameters (tweak as you like)
    N_BLOCKS = 12
    MEAN_Z = 1.0
    STD_Z = 0.15
    SIM_DT = 0.01
    SIM_SECONDS = 8.0

    C = ry.Config()

	# Table
    C.addFrame('table') \
     .setShape(ry.ST.ssBox, size=[1.2, 1.0, 0.08, 0.02]) \
     .setColor([0.3, 0.3, 0.35]) \
     .setPosition([0, 0, 0.4])

	# Blocks
    spawn_blocks(C, n_blocks=N_BLOCKS, mean_z=MEAN_Z, std_z=STD_Z, seed=42)

    C.view(False)

    S = ry.Simulation(C, ry.SimulationEngine.physx, verbose=0)

    steps = int(np.ceil(SIM_SECONDS / SIM_DT))
    for i in range(steps):
        S.step([], SIM_DT, ry.ControlMode.none)

        if i % 2 == 0:
            C.view()
        time.sleep(SIM_DT)

    del S


if __name__ == '__main__':
    main()